<img src="../../../At-Home-Round/Chameleon/figs/IOAI-Logo.png" alt="Logo IOAI" width="200" height="auto">

[IOAI 2025 (Pekin, Chine), manche a domicile](https://ioai-official.org/china-2025)

[![Ouvrir dans Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SKonteye/IOAI-2025/blob/main/fr/At-Home-Round/Chameleon/Chameleon_Solution.ipynb)

## Chargement des donnees


In [ ]:
from IPython.display import display
from PIL import Image
from datasets import Dataset
TRAIN_TEXT = "/bohr/train-7ul9/v2"
hint_description = Dataset.load_from_disk(TRAIN_TEXT + "/dataset/hint_descriptions")
hint_description = {
    x['ID']: {'description': x['Description'], 'icons': x['image']}
    for x in hint_description
}

# exemple
display(hint_description[7]['icons'])
print(hint_description[7]['description'])

In [ ]:
from torch.utils.data import random_split
import torch
# On suppose que validation_data est un objet Dataset PyTorch
validation_data = Dataset.load_from_disk(TRAIN_TEXT + "/dataset/takehome_validation")
dataset_size = len(validation_data)
validation_size = int(dataset_size * 0.5)
train_size = dataset_size - validation_size
train_data, validation_data = random_split(validation_data, [train_size, validation_size])
print(len(validation_data))

## Implementer le devineur de mots-cles


L'acces a Internet est autorise sur Bohrium, ce qui permet aux concurrents de telecharger des modeles pre-entraines depuis Hugging Face. Cependant, les serveurs etant en Chine continentale, ils sont soumis a des restrictions reseau qui bloquent l'acces a des sources comme huggingface.co. Pour contourner cela, on peut utiliser des miroirs heberges (qui seront egalement fournis lors de la manche sur site). Pour Hugging Face, definissez `os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'` pour utiliser un miroir Hugging Face accessible depuis le serveur de Bohrium.


In [ ]:
import os
os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'

In [ ]:
import logging
from sentence_transformers import SentenceTransformer
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# Configure le logging pour afficher la progression du telechargement
logging.basicConfig(level=logging.INFO)

# Charge le modele
model = SentenceTransformer('sentence-t5-base')

# Encode quelques phrases
embeddings = model.encode([
    'hello world',
    'fun and games'
])

print(f"Embedding shape: {embeddings.shape}")

In [ ]:
def hints_to_sentence(hints: list[int]) -> str:
  sentence = "The following hints at our target word:\n<HINT_PRIMARY>\n"
  for i, hint in enumerate(hints):
    sentence += f"{hint_description[hint]['description']}"
    if i == 0:
      sentence += "\n</HINT_PRIMARY>\n<HINT>\n"
    elif i < len(hints) - 1:
      sentence += "\n</HINT>\n<HINT>\n"
    else:
      sentence += "\n</HINT>"
  return sentence

def choice_to_doc(choice:str)->str:
  return f"{choice}"

print(hints_to_sentence([1, 2, 3]))

In [ ]:
# Fine-tune le modele

from sentence_transformers import SentenceTransformer, InputExample, losses
from sentence_transformers.evaluation import EmbeddingSimilarityEvaluator
from torch.utils.data import DataLoader

import os
os.environ["WANDB_DISABLED"] = "true"

train_examples = []

for val in validation_data:
  train_examples.append(InputExample(texts=[hints_to_sentence(val['hints']), choice_to_doc(val['label'])], label=1))
# Cree le DataLoader
train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=2)

# Definit la fonction de perte
train_loss = losses.CosineSimilarityLoss(model)

model.fit(
    train_objectives=[(train_dataloader, train_loss)],
    epochs=80,
    warmup_steps=5,
    output_path='./model',
    optimizer_params={'lr': 1e-6},
    weight_decay=0.01,
    save_best_model=True,
    show_progress_bar=True
)

In [ ]:
ft_model_loaded = SentenceTransformer("./model") # Charge le modele fine-tune

def find_most_similar(query, sentences, model, top_k=10):
    # Encode la requete et les phrases
    query_embedding = model.encode([query])
    sentence_embeddings = model.encode(sentences)

    # Calcule les similarites
    similarities = cosine_similarity(query_embedding, sentence_embeddings)[0]

    # Recupere les top-k plus similaires
    top_indices = np.argsort(similarities)[::-1][:top_k]

    results = []
    for idx in top_indices:
        results.append({
            'sentence': sentences[idx],
            'similarity': similarities[idx]
        })

    return results

def guess_words(hints: list[int], choices: list[str]) -> list[str]:
    query = hints_to_sentence(hints)
    results = find_most_similar(query, choices, ft_model_loaded)
    return [result['sentence'] for result in results]

In [ ]:
import math

def score(guesses: list[str], gold: str):
    # Normalise en minuscules
    guesses = [g.lower() for g in guesses[:10]]
    gold = gold.lower()

    result = {
        "hits@10": 0.0,
        "ndcg@10": 0.0,
        "total_score": 0.0
    }

    if gold in guesses:
        rank = guesses.index(gold)
        result["hits@10"] = 1.0
        result["ndcg@10"] = 1.0 / math.log2(rank + 2)  # rank + 2 car l'index commence a 0
    else:
        result["hits@10"] = 0.0
        result["ndcg@10"] = 0.0

    result["total_score"] = 0.9 * result["hits@10"] + 0.1 * result["ndcg@10"]
    return result

print(score(['cat', 'dog', 'tree', 'flower', 'rock', 'water', 'fried rice', 'airplane', 'cactus', 'tiger'], gold='cactus'))

In [ ]:
from tqdm.notebook import tqdm

# score sur l'ensemble de validation
guesses = []
total_scores = 0.0
for example in tqdm(validation_data):
    guesses.append(guess_words(example['hints'], example['options']))
    total_scores += score(guesses[-1], example['label'])['total_score']
print(f"Average train score: {total_scores / len(validation_data)}")
guesses = []
total_scores = 0.0
for example in tqdm(train_data):
    guesses.append(guess_words(example['hints'], example['options']))
    total_scores += score(guesses[-1], example['label'])['total_score']
print(f"Average validation score: {total_scores / len(train_data)}")

## Soumission du modele


In [ ]:
model_code = """
from sentence_transformers import SentenceTransformer
from datasets import Dataset
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

TRAIN_TEXT = "/bohr/train-7ul9/v2"
hint_description = Dataset.load_from_disk(TRAIN_TEXT + "/dataset/hint_descriptions")
hint_description = {
    x['ID']: {'description': x['Description'], 'icons': x['image']}
    for x in hint_description
}

model = SentenceTransformer("./model")

def hints_to_sentence(hints: list[int]) -> str:
  sentence = "The following hints at our target word:\\n<HINT_PRIMARY>\\n"
  for i, hint in enumerate(hints):
    sentence += f"{hint_description[hint]['description']}"
    if i == 0:
      sentence += "\\n</HINT_PRIMARY>\\n<HINT>\\n"
    elif i < len(hints) - 1:
      sentence += "\\n</HINT>\\n<HINT>\\n"
    else:
      sentence += "\\n</HINT>"
  return sentence

def choice_to_doc(choice:str)->str:
  return f"Our target word: {choice}"

def find_most_similar(query, sentences, model, top_k=10):
    # Encode la requete et les phrases
    query_embedding = model.encode([query])
    sentence_embeddings = model.encode(sentences)

    # Calcule les similarites
    similarities = cosine_similarity(query_embedding, sentence_embeddings)[0]

    # Recupere les top-k plus similaires
    top_indices = np.argsort(similarities)[::-1][:top_k]

    results = []
    for idx in top_indices:
        results.append({
            'sentence': sentences[idx],
            'similarity': similarities[idx]
        })

    return results

def guess_words(hints: list[int], choices: list[str]) -> list[str]:
  query = hints_to_sentence(hints)
  results = find_most_similar(query, choices, model)
  return [result['sentence'] for result in results]
"""

with open("submission_model.py", "w") as f:
  f.write(model_code)

print("Code d'inference ecrit dans submission_model.py")

In [ ]:
import shutil
import os
import tempfile

# Cree un repertoire temporaire avec la structure souhaitee
with tempfile.TemporaryDirectory() as temp_dir:
    # Copie les fichiers dans le repertoire temporaire
    shutil.copy('submission_model.py', temp_dir)
    shutil.copytree('./model', os.path.join(temp_dir, 'model'))
    
    # Cree l'archive zip
    shutil.make_archive('submission', 'zip', temp_dir)